# Phase 6: SchNet Optuna on Expanded Dataset (MW 200-1000)

## Setup
1. Upload `pyg_3d_graphs_etkdg_expanded.pt` to Google Drive `/MolGap/`
2. Runtime → Change runtime type → GPU (T4)
3. Run all cells
4. Results auto-save to Drive after each trial (断開しても再開可能)

In [ ]:
# Step 0: Install dependencies
import torch
TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER = 'cu' + torch.version.cuda.replace('.', '') if torch.version.cuda else 'cpu'
PYG_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+{CUDA_VER}.html'
print(f'PyTorch: {TORCH_VER}, CUDA tag: {CUDA_VER}')
print(f'PyG wheel URL: {PYG_URL}')

!pip install -q torch-geometric optuna
!pip install -q pyg-lib torch-scatter torch-sparse torch-cluster torch-spline-conv -f {PYG_URL}

print(f'\nGPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE"}')

# Quick sanity check
from torch_geometric.nn.models.schnet import radius_graph
print('radius_graph import OK')

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/MolGap'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')
print('Files:', os.listdir(DRIVE_DIR))

In [ ]:
# Step 2: Define model & training (all inline, no external imports)
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts, ReduceLROnPlateau
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

TARGET_COLS = ['homo', 'lumo', 'gap']
SEED = 42

class SchNetWrapper(nn.Module):
    def __init__(self, hidden_channels, num_filters, num_interactions,
                 num_gaussians, cutoff, dropout=0.1, n_targets=3, use_charges=False):
        super().__init__()
        from torch_geometric.nn.models import SchNet
        self.schnet = SchNet(hidden_channels=hidden_channels, num_filters=num_filters,
            num_interactions=num_interactions, num_gaussians=num_gaussians, cutoff=cutoff)
        self.use_charges = use_charges
        if use_charges:
            self.charge_proj = nn.Linear(1, hidden_channels)
        self.head = nn.Sequential(
            nn.Linear(hidden_channels, hidden_channels), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden_channels, hidden_channels // 2), nn.SiLU(),
            nn.Linear(hidden_channels // 2, n_targets))

    def forward(self, z, pos, batch, charges=None):
        from torch_geometric.nn import global_mean_pool
        from torch_geometric.nn.models.schnet import radius_graph
        h = self.schnet.embedding(z)
        if self.use_charges and charges is not None:
            h = h + self.charge_proj(charges.unsqueeze(-1))
        edge_index = radius_graph(pos, r=self.schnet.cutoff, batch=batch, max_num_neighbors=32)
        row, col = edge_index
        edge_weight = (pos[row] - pos[col]).norm(dim=-1)
        edge_attr = self.schnet.distance_expansion(edge_weight)
        for interaction in self.schnet.interactions:
            h = h + interaction(h, edge_index, edge_weight, edge_attr)
        return self.head(global_mean_pool(h, batch))

def train_epoch(model, loader, optimizer, device, scaler):
    model.train()
    total_loss, n = 0, 0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out = model(batch.z, batch.pos, batch.batch, charges=getattr(batch, 'charges', None))
            loss = F.l1_loss(out, batch.y)
        scaler.scale(loss).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * batch.num_graphs
        n += batch.num_graphs
    return total_loss / n

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds, trues = [], []
    for batch in loader:
        batch = batch.to(device)
        with torch.amp.autocast('cuda'):
            out = model(batch.z, batch.pos, batch.batch, charges=getattr(batch, 'charges', None))
        preds.append(out.cpu().numpy())
        trues.append(batch.y.cpu().numpy())
    return np.vstack(preds), np.vstack(trues)

def run_training(params, train_loader, valid_loader, y_mean, y_std,
                 device, max_epochs, patience, verbose=False, use_charges=False):
    model = SchNetWrapper(
        params['hidden_channels'], params['num_filters'], params['num_interactions'],
        params['num_gaussians'], params['cutoff'], params['dropout'],
        use_charges=use_charges).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
    if params['scheduler'] == 'cosine':
        scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)
    else:
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5,
                                      patience=max(5, patience // 3), min_lr=1e-6)
    scaler = torch.amp.GradScaler('cuda')
    best_val_mae, best_epoch, best_state, log_rows = float('inf'), 0, None, []
    for epoch in range(1, max_epochs + 1):
        t0 = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, device, scaler)
        val_pred, val_true = evaluate(model, valid_loader, device)
        val_mae = float(np.mean(np.abs(val_pred * y_std + y_mean - (val_true * y_std + y_mean))))
        scheduler.step(epoch) if params['scheduler'] == 'cosine' else scheduler.step(val_mae)
        elapsed = time.time() - t0
        log_rows.append({'epoch': epoch, 'train_loss': train_loss, 'val_mae': val_mae,
                         'lr': optimizer.param_groups[0]['lr'], 'time_s': elapsed})
        if val_mae < best_val_mae:
            best_val_mae, best_epoch = val_mae, epoch
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f'  Ep {epoch:3d} | loss={train_loss:.4f} | val_MAE={val_mae:.4f} | '
                  f'best={best_val_mae:.4f}@{best_epoch} | {elapsed:.1f}s')
        if epoch - best_epoch >= patience:
            if verbose: print(f'  Early stop at epoch {epoch} (best={best_epoch})')
            break
    return best_val_mae, best_epoch, best_state, log_rows

def create_split_indices(n, valid_size=0.1, test_size=0.1, random_state=42):
    all_idx = np.arange(n)
    tv, test = train_test_split(all_idx, test_size=test_size, random_state=random_state)
    train, valid = train_test_split(tv, test_size=valid_size/(1-test_size), random_state=random_state)
    return train, valid, test

def regression_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    metrics, maes, rmses, r2s = {}, [], [], []
    for i, t in enumerate(TARGET_COLS):
        mae = float(mean_absolute_error(y_true[:,i], y_pred[:,i]))
        rmse = float(np.sqrt(mean_squared_error(y_true[:,i], y_pred[:,i])))
        r2 = float(r2_score(y_true[:,i], y_pred[:,i]))
        metrics[t] = {'mae': mae, 'rmse': rmse, 'r2': r2}
        maes.append(mae); rmses.append(rmse); r2s.append(r2)
    metrics['average'] = {'mae': np.mean(maes), 'rmse': np.mean(rmses), 'r2': np.mean(r2s)}
    return metrics

print('All definitions ready')

In [ ]:
# Step 3: Load graph data
from torch_geometric.loader import DataLoader

GRAPH_PATH = os.path.join(DRIVE_DIR, 'pyg_3d_graphs_etkdg_expanded.pt')
assert os.path.exists(GRAPH_PATH), f'NOT FOUND: {GRAPH_PATH}\nUpload it to Google Drive /MolGap/ first!'

print('Loading graphs (may take 1-2 min)...')
data_list = torch.load(GRAPH_PATH, weights_only=False)
print(f'Loaded {len(data_list)} graphs')

torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda')

train_idx, valid_idx, test_idx = create_split_indices(len(data_list), random_state=SEED)
train_data = [data_list[i] for i in train_idx]
valid_data = [data_list[i] for i in valid_idx]
test_data  = [data_list[i] for i in test_idx]

train_y = np.stack([d.y.squeeze(0).numpy() for d in train_data])
y_mean = train_y.mean(axis=0)
y_std  = train_y.std(axis=0); y_std[y_std < 1e-6] = 1.0
for d in train_data + valid_data + test_data:
    d.y = (d.y - torch.tensor(y_mean)) / torch.tensor(y_std)

has_charges = hasattr(data_list[0], 'charges')
del data_list
print(f'train={len(train_data)}, valid={len(valid_data)}, test={len(test_data)}, charges={has_charges}')

In [ ]:
# Step 4: Optuna search (auto-resume on disconnect)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 20
FAST_EPOCHS = 100
STUDY_PATH = os.path.join(DRIVE_DIR, 'optuna_phase6.db')
storage = f'sqlite:///{STUDY_PATH}'

# Clean up any failed study from previous attempts
try:
    old = optuna.load_study(study_name='phase6_schnet', storage=storage)
    good = [t for t in old.trials if t.state == optuna.trial.TrialState.COMPLETE and t.value < float('inf')]
    if len(good) == 0:
        print('Previous study has no valid trials, deleting...')
        optuna.delete_study(study_name='phase6_schnet', storage=storage)
    else:
        print(f'Resuming: {len(good)} valid trials found')
except:
    pass

study = optuna.create_study(
    study_name='phase6_schnet', direction='minimize', storage=storage,
    load_if_exists=True, sampler=optuna.samplers.TPESampler(seed=SEED))

completed = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE and t.value < float('inf')])
remaining = max(0, N_TRIALS - completed)
print(f'Completed: {completed}, remaining: {remaining}')

def objective(trial):
    params = {
        'hidden_channels': trial.suggest_categorical('hidden_channels', [128, 192, 256]),
        'num_filters': trial.suggest_categorical('num_filters', [128, 192, 256]),
        'num_interactions': trial.suggest_int('num_interactions', 3, 6),
        'num_gaussians': trial.suggest_categorical('num_gaussians', [25, 50, 100]),
        'cutoff': trial.suggest_float('cutoff', 6.0, 12.0, step=1.0),
        'dropout': trial.suggest_float('dropout', 0.0, 0.3, step=0.05),
        'lr': trial.suggest_float('lr', 1e-4, 2e-3, log=True),
        'weight_decay': trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [32, 64, 128]),
        'scheduler': trial.suggest_categorical('scheduler', ['plateau', 'cosine']),
    }
    train_loader = DataLoader(train_data, batch_size=params['batch_size'], shuffle=True)
    valid_loader = DataLoader(valid_data, batch_size=params['batch_size'])
    try:
        best_mae, best_ep, _, _ = run_training(
            params, train_loader, valid_loader, y_mean, y_std,
            device, max_epochs=FAST_EPOCHS, patience=15, use_charges=has_charges)
    except Exception as e:
        print(f'  Trial {trial.number} FAILED: {e}')
        torch.cuda.empty_cache()
        return float('inf')
    print(f'  Trial {trial.number:2d} | MAE={best_mae:.4f} @ep{best_ep} | '
          f'h={params["hidden_channels"]} int={params["num_interactions"]} '
          f'lr={params["lr"]:.1e} bs={params["batch_size"]}')
    torch.cuda.empty_cache()
    return best_mae

if remaining > 0:
    study.optimize(objective, n_trials=remaining)

best = study.best_trial
print(f'\nBest trial {best.number}: MAE={best.value:.4f}')
print(f'Params: {best.params}')

In [ ]:
# Step 5: Full retrain with best params
best_params = dict(study.best_trial.params)
print(f'Retraining with best params...')

train_loader = DataLoader(train_data, batch_size=best_params['batch_size'], shuffle=True)
valid_loader = DataLoader(valid_data, batch_size=best_params['batch_size'])
test_loader  = DataLoader(test_data,  batch_size=best_params['batch_size'])

best_mae, best_epoch, best_state, log_rows = run_training(
    best_params, train_loader, valid_loader, y_mean, y_std,
    device, max_epochs=500, patience=40, verbose=True, use_charges=has_charges)

model = SchNetWrapper(
    best_params['hidden_channels'], best_params['num_filters'],
    best_params['num_interactions'], best_params['num_gaussians'],
    best_params['cutoff'], best_params['dropout'], use_charges=has_charges).to(device)
model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

test_pred, test_true = evaluate(model, test_loader, device)
m = regression_metrics(test_pred * y_std + y_mean, test_true * y_std + y_mean)

print(f'\n{"="*60}')
print(f'  Phase 6 SchNet Optuna (MW 200-1000) Test Results')
print(f'{"="*60}')
for t in TARGET_COLS:
    print(f'  {t:5s}: MAE={m[t]["mae"]:.4f}  R2={m[t]["r2"]:.4f}')
print(f'  avg  : MAE={m["average"]["mae"]:.4f}  R2={m["average"]["r2"]:.4f}')

In [ ]:
# Step 6: Save everything to Drive
import pandas as pd

n_total = len(train_data) + len(valid_data) + len(test_data)

torch.save(best_state, os.path.join(DRIVE_DIR, 'gnn_schnet_3d_optuna_expanded.pt'))

with open(os.path.join(DRIVE_DIR, 'optuna_best_params_phase6.json'), 'w') as f:
    json.dump(best_params, f, indent=2)

with open(os.path.join(DRIVE_DIR, 'schnet_optuna_expanded_metrics.json'), 'w') as f:
    json.dump({'model': 'SchNet_3D_ETKDG_optuna_expanded', 'params': best_params,
               'n_data': n_total, 'mw_range': '200-1000',
               'best_epoch': best_epoch, 'epochs_trained': len(log_rows),
               'metrics': m}, f, indent=2)

pd.DataFrame(log_rows).to_csv(os.path.join(DRIVE_DIR, 'retrain_log_phase6.csv'), index=False)
study.trials_dataframe().to_csv(os.path.join(DRIVE_DIR, 'optuna_trials_phase6.csv'), index=False)

print(f'All saved to {DRIVE_DIR}/')
for f in sorted(os.listdir(DRIVE_DIR)):
    size = os.path.getsize(os.path.join(DRIVE_DIR, f)) / 1024 / 1024
    print(f'  {f} ({size:.1f} MB)')